# 03 Feature Engineering

This notebook constructs the pre-match features used for predictive modelling. Features are built chronologically to prevent data leakage and include ranking-based indicators and rolling serve/return performance metrics.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Load Cleaned Data

The cleaned match, ranking, and player datasets produced in the preprocessing stage are loaded here for feature engineering.

In [2]:
DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
FEATURE_DIR = DATA_DIR / "features"
FEATURE_DIR.mkdir(exist_ok=True)

df_matches = pd.read_csv(PROCESSED_DIR / "matches_clean.csv", parse_dates=["tourney_date"])
df_rankings = pd.read_csv(PROCESSED_DIR / "rankings_clean.csv", parse_dates=["ranking_date"])
df_players = pd.read_csv(PROCESSED_DIR / "players_clean.csv", parse_dates=["dob"])

print("Matches:", df_matches.shape)
print("Rankings:", df_rankings.shape)
print("Players:", df_players.shape)

Matches: (29408, 49)
Rankings: (1340288, 4)
Players: (65989, 8)


/var/folders/wv/tbs_c8x12sg2k57wmnt893b80000gn/T/ipykernel_35084/2184620461.py:8: DtypeWarning: Columns (0: wikidata_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df_players = pd.read_csv(PROCESSED_DIR / "players_clean.csv", parse_dates=["dob"])


## 2. Transform Match Data into Player-Match Format

The cleaned match dataset is stored in winner-loser format, with one row per match. For rolling feature construction, this is transformed into a player-match dataset containing one row per player per match.

This long-format representation makes it possible to construct chronological player histories such as last-15-match and 52-week rolling performance indicators.

In [3]:
base_cols = [
    "tourney_id",
    "tourney_name",
    "surface",
    "draw_size",
    "tourney_level",
    "tourney_date",
    "match_num",
    "score",
    "best_of",
    "round",
    "minutes"
]

winner_cols = {
    "winner_id": "player_id",
    "winner_name": "player_name",
    "winner_hand": "player_hand",
    "winner_ht": "player_ht",
    "winner_age": "player_age",
    "loser_id": "opponent_id",
    "loser_name": "opponent_name",
    "loser_hand": "opponent_hand",
    "loser_ht": "opponent_ht",
    "loser_age": "opponent_age",
    "winner_rank": "player_rank",
    "winner_rank_points": "player_rank_points",
    "loser_rank": "opponent_rank",
    "loser_rank_points": "opponent_rank_points",
    "w_ace": "ace",
    "w_df": "df",
    "w_svpt": "svpt",
    "w_1stIn": "first_in",
    "w_1stWon": "first_won",
    "w_2ndWon": "second_won",
    "w_SvGms": "service_games",
    "w_bpSaved": "bp_saved",
    "w_bpFaced": "bp_faced",
    "l_ace": "opp_ace",
    "l_df": "opp_df",
    "l_svpt": "opp_svpt",
    "l_1stIn": "opp_first_in",
    "l_1stWon": "opp_first_won",
    "l_2ndWon": "opp_second_won",
    "l_SvGms": "opp_service_games",
    "l_bpSaved": "opp_bp_saved",
    "l_bpFaced": "opp_bp_faced",
}

loser_cols = {
    "loser_id": "player_id",
    "loser_name": "player_name",
    "loser_hand": "player_hand",
    "loser_ht": "player_ht",
    "loser_age": "player_age",
    "winner_id": "opponent_id",
    "winner_name": "opponent_name",
    "winner_hand": "opponent_hand",
    "winner_ht": "opponent_ht",
    "winner_age": "opponent_age",
    "loser_rank": "player_rank",
    "loser_rank_points": "player_rank_points",
    "winner_rank": "opponent_rank",
    "winner_rank_points": "opponent_rank_points",
    "l_ace": "ace",
    "l_df": "df",
    "l_svpt": "svpt",
    "l_1stIn": "first_in",
    "l_1stWon": "first_won",
    "l_2ndWon": "second_won",
    "l_SvGms": "service_games",
    "l_bpSaved": "bp_saved",
    "l_bpFaced": "bp_faced",
    "w_ace": "opp_ace",
    "w_df": "opp_df",
    "w_svpt": "opp_svpt",
    "w_1stIn": "opp_first_in",
    "w_1stWon": "opp_first_won",
    "w_2ndWon": "opp_second_won",
    "w_SvGms": "opp_service_games",
    "w_bpSaved": "opp_bp_saved",
    "w_bpFaced": "opp_bp_faced",
}

winner_df = df_matches[base_cols + list(winner_cols.keys())].rename(columns=winner_cols).copy()
winner_df["won"] = 1

loser_df = df_matches[base_cols + list(loser_cols.keys())].rename(columns=loser_cols).copy()
loser_df["won"] = 0

df_long = pd.concat([winner_df, loser_df], ignore_index=True)

df_long = df_long.sort_values(
    by=["player_id", "tourney_date", "tourney_id", "match_num"]
).reset_index(drop=True)

print("Long-format player-match shape:", df_long.shape)
print("player_id in columns?", "player_id" in df_long.columns)
df_long.head()

Long-format player-match shape: (58816, 44)
player_id in columns? True


,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,score,best_of,round,...,opp_ace,opp_df,opp_svpt,opp_first_in,opp_first_won,opp_second_won,opp_service_games,opp_bp_saved,opp_bp_faced,won
0,2013-414,Hamburg,Clay,48,A,2013-07-15,6,6-3 6-2,3,R64,...,2.0,0.0,56.0,42.0,25.0,9.0,9.0,6.0,9.0,0
1,2014-308,Munich,Clay,28,A,2014-04-28,4,6-1 6-2,3,R32,...,2.0,4.0,47.0,30.0,24.0,9.0,8.0,6.0,7.0,0
2,2014-321,Stuttgart,Clay,28,A,2014-07-07,8,7-6(7) 7-6(9),3,R32,...,5.0,1.0,86.0,54.0,43.0,16.0,12.0,1.0,3.0,0
3,2014-414,Hamburg,Clay,48,A,2014-07-14,8,6-0 6-2,3,R64,...,1.0,0.0,52.0,34.0,17.0,5.0,7.0,4.0,9.0,1
4,2014-414,Hamburg,Clay,48,A,2014-07-14,24,7-5 7-5,3,R32,...,1.0,4.0,86.0,46.0,25.0,21.0,12.0,7.0,11.0,1


## 3. Derive Per-Match Serve and Return Metrics

Per-match serving and returning variables are derived from the player-match table. These intermediate metrics are later aggregated into rolling historical features.

In [4]:
# Serve metrics
df_long["ace_rate"] = df_long["ace"] / df_long["svpt"]
df_long["df_rate"] = df_long["df"] / df_long["svpt"]

df_long["first_serve_in_pct"] = df_long["first_in"] / df_long["svpt"]
df_long["first_serve_win_pct"] = df_long["first_won"] / df_long["first_in"]

second_serve_points = df_long["svpt"] - df_long["first_in"]
df_long["second_serve_win_pct"] = np.where(
    second_serve_points > 0,
    df_long["second_won"] / second_serve_points,
    np.nan
)

df_long["service_points_won_pct"] = (
    df_long["first_won"] + df_long["second_won"]
) / df_long["svpt"]

df_long["bp_saved_pct"] = np.where(
    df_long["bp_faced"] > 0,
    df_long["bp_saved"] / df_long["bp_faced"],
    np.nan
)

df_long["bp_faced_rate"] = np.where(
    df_long["service_games"] > 0,
    df_long["bp_faced"] / df_long["service_games"],
    np.nan
)

# Return metrics
df_long["return_points_won_pct"] = np.where(
    df_long["opp_svpt"] > 0,
    (df_long["opp_svpt"] - (df_long["opp_first_won"] + df_long["opp_second_won"])) / df_long["opp_svpt"],
    np.nan
)

df_long["bp_converted"] = df_long["opp_bp_faced"] - df_long["opp_bp_saved"]

df_long["bp_converted_pct"] = np.where(
    df_long["opp_bp_faced"] > 0,
    df_long["bp_converted"] / df_long["opp_bp_faced"],
    np.nan
)

df_long["bp_earned_rate"] = np.where(
    df_long["opp_service_games"] > 0,
    df_long["opp_bp_faced"] / df_long["opp_service_games"],
    np.nan
)

metric_cols = [
    "ace_rate",
    "df_rate",
    "first_serve_in_pct",
    "first_serve_win_pct",
    "second_serve_win_pct",
    "service_points_won_pct",
    "bp_saved_pct",
    "bp_faced_rate",
    "return_points_won_pct",
    "bp_converted_pct",
    "bp_earned_rate",
]

df_long[metric_cols].describe()

,ace_rate,df_rate,first_serve_in_pct,first_serve_win_pct,second_serve_win_pct,service_points_won_pct,bp_saved_pct,bp_faced_rate,return_points_won_pct,bp_converted_pct,bp_earned_rate
count,58816.000000,58816.000000,58816.000000,58816.000000,58814.000000,58816.000000,55967.000000,58816.000000,58816.000000,55967.000000,58816.000000
mean,0.079418,0.036694,0.616047,0.718770,0.512339,0.638561,0.589402,0.543121,0.361439,0.410598,0.543121
std,0.062077,0.027339,0.076821,0.099412,0.115385,0.088068,0.249617,0.346382,0.088068,0.249617,0.346382
min,0.000000,0.000000,0.270270,0.105263,0.000000,0.218750,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.034483,0.017094,0.565217,0.654545,0.437500,0.581081,0.454545,0.291667,0.301887,0.250000,0.291667
50%,0.065789,0.032363,0.616822,0.722222,0.512195,0.640000,0.600000,0.500000,0.360000,0.400000,0.500000
75%,0.109375,0.051724,0.666667,0.787879,0.586207,0.698113,0.750000,0.750000,0.418919,0.545455,0.750000
max,0.812500,0.307692,1.000000,1.000000,1.615385,1.000000,1.000000,3.000000,0.781250,1.000000,3.000000


## 4. Construct Rolling Historical Performance Features

Rolling features are created to represent each player's recent and medium-term form prior to a match. Two windows are used:

- Last 15 matches (L15), capturing short-term form
- Previous 52 weeks (52W), capturing medium-term performance

To prevent data leakage, all rolling features are computed using only matches that occurred before the current match.

In [5]:
rolling_metric_map = {
    "ace_rate": "AceRate",
    "df_rate": "DFRate",
    "first_serve_in_pct": "FirstServeInPct",
    "first_serve_win_pct": "FirstServeWinPct",
    "second_serve_win_pct": "SecondServeWinPct",
    "service_points_won_pct": "ServicePointsWonPct",
    "bp_saved_pct": "BreakPointsSavedPct",
    "bp_faced_rate": "BreakPointsFacedRate",
    "return_points_won_pct": "ReturnPointsWonPct",
    "bp_converted_pct": "BreakPointsConvertedPct",
    "bp_earned_rate": "BreakPointsEarnedRate",
}

df_long = df_long.sort_values(
    ["player_id", "tourney_date", "tourney_id", "match_num"]
).reset_index(drop=True)

# L15 features
for metric, prefix in rolling_metric_map.items():
    df_long[f"{prefix}L15"] = (
        df_long.groupby("player_id")[metric]
        .transform(lambda s: s.shift(1).rolling(window=15, min_periods=1).mean())
    )

l15_cols = [f"{prefix}L15" for prefix in rolling_metric_map.values()]

In [6]:
# 52W features built player-by-player to avoid groupby/apply issues

w52_parts = []

for pid, group in df_long.groupby("player_id"):
    group = group.sort_values(["tourney_date", "tourney_id", "match_num"]).copy()
    group = group.set_index("tourney_date")

    for metric, prefix in rolling_metric_map.items():
        group[f"{prefix}52W"] = (
            group[metric]
            .shift(1)
            .rolling("365D", min_periods=1)
            .mean()
        )

    group = group.reset_index()
    group["player_id"] = pid
    w52_parts.append(group)

df_long = pd.concat(w52_parts, ignore_index=True)

df_long = df_long.sort_values(
    ["player_id", "tourney_date", "tourney_id", "match_num"]
).reset_index(drop=True)

w52_cols = [f"{prefix}52W" for prefix in rolling_metric_map.values()]

print("player_id in columns?", "player_id" in df_long.columns)
print("df_long shape:", df_long.shape)

display(df_long[l15_cols + w52_cols].describe())

player_id in columns? True
df_long shape: (58816, 78)


,AceRateL15,DFRateL15,FirstServeInPctL15,FirstServeWinPctL15,SecondServeWinPctL15,ServicePointsWonPctL15,BreakPointsSavedPctL15,BreakPointsFacedRateL15,ReturnPointsWonPctL15,BreakPointsConvertedPctL15,...,DFRate52W,FirstServeInPct52W,FirstServeWinPct52W,SecondServeWinPct52W,ServicePointsWonPct52W,BreakPointsSavedPct52W,BreakPointsFacedRate52W,ReturnPointsWonPct52W,BreakPointsConvertedPct52W,BreakPointsEarnedRate52W
count,57970.000000,57970.000000,57970.000000,57970.000000,57970.000000,57970.000000,57956.000000,57970.000000,57970.000000,57929.000000,...,57970.000000,57970.000000,57970.000000,57970.000000,57970.000000,57955.000000,57970.000000,57970.000000,57888.000000,57970.000000
mean,0.079446,0.036588,0.615877,0.719310,0.513571,0.639350,0.591549,0.540463,0.363050,0.411549,...,0.036545,0.615538,0.718012,0.512589,0.638144,0.589841,0.543935,0.362900,0.410955,0.548371
std,0.043894,0.014339,0.043336,0.051224,0.043093,0.042275,0.082594,0.146617,0.037213,0.080609,...,0.013410,0.040993,0.049022,0.039735,0.040070,0.071878,0.138105,0.035129,0.074042,0.123185
min,0.000000,0.000000,0.333333,0.272727,0.133333,0.300000,0.000000,0.000000,0.085714,0.000000,...,0.000000,0.333333,0.272727,0.111111,0.300000,0.000000,0.000000,0.085714,0.000000,0.000000
25%,0.048553,0.026584,0.586856,0.685337,0.486811,0.611734,0.541106,0.444962,0.339933,0.363443,...,0.027327,0.589158,0.687690,0.492126,0.613770,0.553863,0.457207,0.343440,0.377879,0.477244
50%,0.070804,0.034653,0.615888,0.719467,0.513279,0.637924,0.591966,0.538855,0.362433,0.412287,...,0.034461,0.615756,0.719417,0.513861,0.637960,0.593671,0.540972,0.364077,0.414957,0.550565
75%,0.100525,0.044112,0.644619,0.753965,0.540471,0.666039,0.642593,0.632126,0.386086,0.460835,...,0.043555,0.642514,0.750908,0.535775,0.663244,0.630556,0.620306,0.383690,0.449123,0.620814
max,0.321429,0.179487,0.842105,0.944444,0.875000,0.837209,1.000000,2.375000,0.666667,1.000000,...,0.179487,0.880000,0.944444,0.875000,0.837209,1.000000,2.375000,0.666667,1.000000,2.142857


In [7]:
display(
    df_long[
        ["player_id", "player_name", "tourney_date", "won"] + w52_cols[:5]
    ].head(20)
)

,player_id,player_name,tourney_date,won,AceRate52W,DFRate52W,FirstServeInPct52W,FirstServeWinPct52W,SecondServeWinPct52W
0,100644,Alexander Zverev,2013-07-15,0,NaN,NaN,NaN,NaN,NaN
1,100644,Alexander Zverev,2014-04-28,0,0.025641,0.179487,0.358974,0.500000,0.240000
2,100644,Alexander Zverev,2014-07-07,0,0.024184,0.101107,0.406760,0.475000,0.286667
3,100644,Alexander Zverev,2014-07-14,1,0.037628,0.096079,0.464722,0.532716,0.396239
4,100644,Alexander Zverev,2014-07-14,1,0.045267,0.077741,0.490587,0.569537,0.494548
5,100644,Alexander Zverev,2014-07-14,1,0.036213,0.077777,0.485976,0.594519,0.507833
6,100644,Alexander Zverev,2014-07-14,1,0.043903,0.068736,0.506941,0.607612,0.519154
7,100644,Alexander Zverev,2014-07-14,0,0.048084,0.065885,0.511176,0.621459,0.501380
8,100644,Alexander Zverev,2014-07-21,0,0.051449,0.067024,0.503529,0.606277,0.472799
9,100644,Alexander Zverev,2014-07-27,0,0.059377,0.067375,0.513858,0.607540,0.468575


## 5. Construct Ranking-Based Features

Ranking-based features are used to capture both current player strength and broader performance trajectory over time. In addition to the current match-time ranking, engineered ranking features include career-high ranking, weeks since career-high, 26-week ranking change, and 52-week ranking volatility.

These features are derived from weekly ATP ranking history rather than match rows, as ranking is updated on a calendar basis rather than after each match.

In [8]:
rank_weekly = df_rankings.copy()

rank_weekly = rank_weekly.rename(
    columns={
        "player": "player_id",
        "rank": "weekly_rank",
        "points": "weekly_points",
    }
)

rank_weekly = rank_weekly.sort_values(["player_id", "ranking_date"]).reset_index(drop=True)

# Career-high
rank_weekly["RankingCareerHigh"] = (
    rank_weekly.groupby("player_id")["weekly_rank"].cummin()
)

new_career_high = rank_weekly["weekly_rank"].eq(rank_weekly["RankingCareerHigh"])

rank_weekly["career_high_date"] = (
    rank_weekly["ranking_date"]
    .where(new_career_high)
    .groupby(rank_weekly["player_id"])
    .ffill()
)

# 26-week change
rank_weekly["target_26w_date"] = rank_weekly["ranking_date"] - pd.Timedelta(weeks=26)

history_26w = rank_weekly[["player_id", "ranking_date", "weekly_rank"]].rename(
    columns={
        "ranking_date": "ranking_date_26w",
        "weekly_rank": "rank_26w_ago",
    }
)

rank_change_parts = []

for pid, group in rank_weekly.groupby("player_id"):
    current_g = group.sort_values("target_26w_date").copy()
    history_g = history_26w[history_26w["player_id"] == pid].sort_values("ranking_date_26w").copy()

    merged_g = pd.merge_asof(
        current_g,
        history_g[["ranking_date_26w", "rank_26w_ago"]],
        left_on="target_26w_date",
        right_on="ranking_date_26w",
        direction="backward"
    )

    rank_change_parts.append(merged_g)

rank_weekly = pd.concat(rank_change_parts, ignore_index=True)
rank_weekly = rank_weekly.sort_values(["player_id", "ranking_date"]).reset_index(drop=True)

rank_weekly["RankChange26W"] = rank_weekly["weekly_rank"] - rank_weekly["rank_26w_ago"]

# 52-week std
rank_std_52w = (
    rank_weekly.groupby("player_id")
    .rolling("365D", on="ranking_date", min_periods=2)["weekly_rank"]
    .std()
    .reset_index()
    .rename(columns={"weekly_rank": "RankSTD52W"})
)

rank_weekly = rank_weekly.merge(
    rank_std_52w,
    on=["player_id", "ranking_date"],
    how="left"
)

rank_weekly.head()

,ranking_date,weekly_rank,player_id,weekly_points,RankingCareerHigh,career_high_date,target_26w_date,ranking_date_26w,rank_26w_ago,RankChange26W,RankSTD52W
0,2012-08-27,1537,100644,1.0,1537,2012-08-27,2012-02-27,NaT,NaN,NaN,NaN
1,2012-09-10,1392,100644,2.0,1392,2012-09-10,2012-03-12,NaT,NaN,NaN,102.530483
2,2012-09-17,1397,100644,2.0,1392,2012-09-10,2012-03-19,NaT,NaN,NaN,82.310388
3,2012-09-24,1406,100644,2.0,1392,2012-09-10,2012-03-26,NaT,NaN,NaN,69.574900
4,2012-10-01,1409,100644,2.0,1392,2012-09-10,2012-04-02,NaT,NaN,NaN,61.202124


## 6.1 Merge Weekly Ranking Features into the Player-Match Table

For each player-match row, the most recent available ATP ranking record on or before the match date is merged into the long-format dataset. This ensures that all ranking-based features reflect information available prior to the match.

In [9]:
rank_feature_cols = [
    "player_id",
    "ranking_date",
    "weekly_rank",
    "weekly_points",
    "RankingCareerHigh",
    "career_high_date",
    "rank_26w_ago",
    "RankChange26W",
    "RankSTD52W",
]

rank_weekly_merge = rank_weekly[rank_feature_cols].copy()
rank_weekly_merge = rank_weekly_merge.sort_values(["player_id", "ranking_date"]).reset_index(drop=True)

df_long = df_long.sort_values(["player_id", "tourney_date", "tourney_id", "match_num"]).reset_index(drop=True)

merged_parts = []

for pid, matches_g in df_long.groupby("player_id"):
    matches_g = matches_g.sort_values("tourney_date").copy()

    ranks_g = rank_weekly_merge[
        rank_weekly_merge["player_id"] == pid
    ].sort_values("ranking_date").copy()

    merged_g = pd.merge_asof(
        matches_g,
        ranks_g.drop(columns=["player_id"]),
        left_on="tourney_date",
        right_on="ranking_date",
        direction="backward"
    )

    merged_parts.append(merged_g)

df_long = pd.concat(merged_parts, ignore_index=True)
df_long = df_long.sort_values(["player_id", "tourney_date", "tourney_id", "match_num"]).reset_index(drop=True)

print("df_long shape after ranking merge:", df_long.shape)

display(
    df_long[
        [
            "player_id",
            "player_name",
            "tourney_date",
            "player_rank",
            "weekly_rank",
            "RankingCareerHigh",
            "career_high_date",
            "RankChange26W",
            "RankSTD52W",
        ]
    ].head(20)
)

df_long shape after ranking merge: (58816, 86)


,player_id,player_name,tourney_date,player_rank,weekly_rank,RankingCareerHigh,career_high_date,RankChange26W,RankSTD52W
0,100644,Alexander Zverev,2013-07-15,798.0,798.0,783.0,2013-06-24,-41.0,235.576729
1,100644,Alexander Zverev,2014-04-28,765.0,765.0,694.0,2013-11-11,-1.0,34.948581
2,100644,Alexander Zverev,2014-07-07,285.0,285.0,285.0,2014-07-07,-523.0,87.471317
3,100644,Alexander Zverev,2014-07-14,285.0,285.0,285.0,2014-07-14,-522.0,112.122492
4,100644,Alexander Zverev,2014-07-14,285.0,285.0,285.0,2014-07-14,-522.0,112.122492
5,100644,Alexander Zverev,2014-07-14,285.0,285.0,285.0,2014-07-14,-522.0,112.122492
6,100644,Alexander Zverev,2014-07-14,285.0,285.0,285.0,2014-07-14,-522.0,112.122492
7,100644,Alexander Zverev,2014-07-14,285.0,285.0,285.0,2014-07-14,-522.0,112.122492
8,100644,Alexander Zverev,2014-07-21,161.0,161.0,161.0,2014-07-21,-646.0,141.603801
9,100644,Alexander Zverev,2014-07-27,159.0,161.0,161.0,2014-07-21,-646.0,141.603801


## 6.2 Create Final Ranking Feature Columns

The merged weekly ranking history is used to construct the final ranking-based features used for modelling, including current ranking, career-high ranking, weeks since career-high, 26-week ranking change, and 52-week ranking volatility.

In [10]:
df_long["CurrentRanking"] = df_long["weekly_rank"].fillna(df_long["player_rank"])

df_long["WeekSinceCareerHigh"] = (
    (df_long["tourney_date"] - df_long["career_high_date"]).dt.days / 7
)

ranking_feature_cols = [
    "CurrentRanking",
    "RankingCareerHigh",
    "WeekSinceCareerHigh",
    "RankChange26W",
    "RankSTD52W",
]

print(df_long[ranking_feature_cols].describe())

display(
    df_long[
        [
            "player_name",
            "tourney_date",
            "player_rank",
            "CurrentRanking",
            "RankingCareerHigh",
            "WeekSinceCareerHigh",
            "RankChange26W",
            "RankSTD52W",
        ]
    ].head(20)
)

       CurrentRanking  RankingCareerHigh  WeekSinceCareerHigh  RankChange26W  \
count    58802.000000       58802.000000         58802.000000   58779.000000   
mean        73.984337          45.411517            79.598053     -11.612413   
std         99.753966          69.843843            96.994623      70.502765   
min          1.000000           1.000000             0.000000   -1608.000000   
25%         24.000000          11.000000             5.000000     -20.000000   
50%         52.000000          29.000000            38.000000      -2.000000   
75%         90.000000          55.000000           123.000000       6.000000   
max       2159.000000        2048.000000           656.000000    1300.000000   

         RankSTD52W  
count  58802.000000  
mean      21.620715  
std       39.211536  
min        0.000000  
25%        4.749702  
50%       10.941899  
75%       22.296092  
max      695.391419  


,player_name,tourney_date,player_rank,CurrentRanking,RankingCareerHigh,WeekSinceCareerHigh,RankChange26W,RankSTD52W
0,Alexander Zverev,2013-07-15,798.0,798.0,783.0,3.000000,-41.0,235.576729
1,Alexander Zverev,2014-04-28,765.0,765.0,694.0,24.000000,-1.0,34.948581
2,Alexander Zverev,2014-07-07,285.0,285.0,285.0,0.000000,-523.0,87.471317
3,Alexander Zverev,2014-07-14,285.0,285.0,285.0,0.000000,-522.0,112.122492
4,Alexander Zverev,2014-07-14,285.0,285.0,285.0,0.000000,-522.0,112.122492
5,Alexander Zverev,2014-07-14,285.0,285.0,285.0,0.000000,-522.0,112.122492
6,Alexander Zverev,2014-07-14,285.0,285.0,285.0,0.000000,-522.0,112.122492
7,Alexander Zverev,2014-07-14,285.0,285.0,285.0,0.000000,-522.0,112.122492
8,Alexander Zverev,2014-07-21,161.0,161.0,161.0,0.000000,-646.0,141.603801
9,Alexander Zverev,2014-07-27,159.0,161.0,161.0,0.857143,-646.0,141.603801


## 6.3 Save Intermediate Feature Table

The long-format player-match table with rolling serve/return and ranking-based features is saved as an intermediate output before Elo feature construction.

In [11]:
df_long.to_csv(FEATURE_DIR / "player_match_features_pre_elo.csv", index=False)

print("Saved:")
print((FEATURE_DIR / "player_match_features_pre_elo.csv").resolve())

Saved:
/Users/baileyhorsley/Documents/C. University/3rd Year/Project/Dissertation_Project/data/features/player_match_features_pre_elo.csv


## 7. Construct Elo Rating Features

Elo ratings are used to model player strength dynamically over time based on match outcomes. Unlike official ATP rankings, Elo ratings are updated sequentially after every match and therefore provide a flexible estimate of player ability that responds more quickly to recent performance.

In this study, Elo ratings are calculated chronologically so that only information available before each match is used when constructing features.

In [12]:
elo_matches = df_matches.copy()

elo_matches = elo_matches.sort_values(
    ["tourney_date", "tourney_id", "match_num"]
).reset_index(drop=True)

print(elo_matches.shape)
elo_matches[["tourney_date", "winner_name", "loser_name", "surface"]].head()

(29408, 49)


,tourney_date,winner_name,loser_name,surface
0,2013-01-07,Yen Hsun Lu,Benoit Paire,Hard
1,2013-01-07,Lukas Lacko,Paolo Lorenzi,Hard
2,2013-01-07,Thomaz Bellucci,David Goffin,Hard
3,2013-01-07,Igor Sijsling,Robin Haase,Hard
4,2013-01-07,Gael Monfils,Benjamin Becker,Hard


In [13]:
def expected_score(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

def update_elo(rating, expected, actual, k=32):
    return rating + k * (actual - expected)

In [14]:
from collections import defaultdict

# Default starting Elo
INITIAL_ELO = 1500
K_FACTOR = 32

# Store current Elo ratings
elo_ratings = defaultdict(lambda: INITIAL_ELO)

# Store career-high Elo
elo_highs = defaultdict(lambda: INITIAL_ELO)

# Lists to attach to match rows
winner_elo_pre = []
loser_elo_pre = []
winner_elo_post = []
loser_elo_post = []
winner_elo_high_pre = []
loser_elo_high_pre = []

for _, row in elo_matches.iterrows():
    winner_id = row["winner_id"]
    loser_id = row["loser_id"]

    # Pre-match ratings
    w_elo = elo_ratings[winner_id]
    l_elo = elo_ratings[loser_id]

    winner_elo_pre.append(w_elo)
    loser_elo_pre.append(l_elo)

    winner_elo_high_pre.append(elo_highs[winner_id])
    loser_elo_high_pre.append(elo_highs[loser_id])

    # Expected scores
    w_expected = expected_score(w_elo, l_elo)
    l_expected = expected_score(l_elo, w_elo)

    # Actual result: winner=1, loser=0
    w_new = update_elo(w_elo, w_expected, actual=1, k=K_FACTOR)
    l_new = update_elo(l_elo, l_expected, actual=0, k=K_FACTOR)

    winner_elo_post.append(w_new)
    loser_elo_post.append(l_new)

    # Update ratings
    elo_ratings[winner_id] = w_new
    elo_ratings[loser_id] = l_new

    # Update career-high Elo
    elo_highs[winner_id] = max(elo_highs[winner_id], w_new)
    elo_highs[loser_id] = max(elo_highs[loser_id], l_new)

# Attach to match dataframe
elo_matches["winner_ELO"] = winner_elo_pre
elo_matches["loser_ELO"] = loser_elo_pre

elo_matches["winner_ELO_post"] = winner_elo_post
elo_matches["loser_ELO_post"] = loser_elo_post

elo_matches["winner_ELOHigh"] = winner_elo_high_pre
elo_matches["loser_ELOHigh"] = loser_elo_high_pre

In [15]:
elo_matches[
    [
        "tourney_date",
        "winner_name",
        "loser_name",
        "winner_ELO",
        "loser_ELO",
        "winner_ELO_post",
        "loser_ELO_post",
        "winner_ELOHigh",
        "loser_ELOHigh",
    ]
].head(20)

,tourney_date,winner_name,loser_name,winner_ELO,loser_ELO,winner_ELO_post,loser_ELO_post,winner_ELOHigh,loser_ELOHigh
0,2013-01-07,Yen Hsun Lu,Benoit Paire,1500.000000,1500.0,1516.000000,1484.000000,1500.000000,1500.0
1,2013-01-07,Lukas Lacko,Paolo Lorenzi,1500.000000,1500.0,1516.000000,1484.000000,1500.000000,1500.0
2,2013-01-07,Thomaz Bellucci,David Goffin,1500.000000,1500.0,1516.000000,1484.000000,1500.000000,1500.0
3,2013-01-07,Igor Sijsling,Robin Haase,1500.000000,1500.0,1516.000000,1484.000000,1500.000000,1500.0
4,2013-01-07,Gael Monfils,Benjamin Becker,1500.000000,1500.0,1516.000000,1484.000000,1500.000000,1500.0
5,2013-01-07,Greg Jones,Jurgen Melzer,1500.000000,1500.0,1516.000000,1484.000000,1500.000000,1500.0
6,2013-01-07,Brian Baker,Jerzy Janowicz,1500.000000,1500.0,1516.000000,1484.000000,1500.000000,1500.0
7,2013-01-07,Jesse Levine,Daniel King Turner,1500.000000,1500.0,1516.000000,1484.000000,1500.000000,1500.0
8,2013-01-07,Olivier Rochus,Albert Ramos,1500.000000,1500.0,1516.000000,1484.000000,1500.000000,1500.0
9,2013-01-07,Xavier Malisse,Martin Klizan,1500.000000,1500.0,1516.000000,1484.000000,1500.000000,1500.0


In [16]:
elo_matches[
    ["winner_ELO", "loser_ELO", "winner_ELOHigh", "loser_ELOHigh"]
].describe()

,winner_ELO,loser_ELO,winner_ELOHigh,loser_ELOHigh
count,29408.000000,29408.000000,29408.000000,29408.000000
mean,1660.118818,1592.346508,1729.091748,1660.220010
std,166.098435,128.304565,176.554488,142.047831
min,1291.732288,1296.282898,1500.000000,1500.000000
25%,1533.232425,1499.997759,1590.722210,1536.745266
50%,1627.083527,1566.788657,1698.154509,1637.886913
75%,1755.636862,1659.391922,1835.243197,1740.481023
max,2293.261910,2297.185069,2297.185069,2297.185069


## 7.1 Construct Surface-Specific Elo Ratings

In addition to overall Elo, surface-specific Elo ratings are calculated for hard, clay, and grass courts. These ratings are updated only when a player competes on the corresponding surface, allowing the model to capture surface-dependent performance differences.

In [17]:
from collections import defaultdict

INITIAL_ELO = 1500
K_FACTOR = 32

surface_elo = {
    "Hard": defaultdict(lambda: INITIAL_ELO),
    "Clay": defaultdict(lambda: INITIAL_ELO),
    "Grass": defaultdict(lambda: INITIAL_ELO),
}

winner_surface_elo_pre = []
loser_surface_elo_pre = []

for _, row in elo_matches.iterrows():
    surface = row["surface"]
    winner_id = row["winner_id"]
    loser_id = row["loser_id"]

    # Only compute for the three main surfaces
    if surface in surface_elo:
        ratings = surface_elo[surface]

        w_elo = ratings[winner_id]
        l_elo = ratings[loser_id]

        winner_surface_elo_pre.append(w_elo)
        loser_surface_elo_pre.append(l_elo)

        w_expected = expected_score(w_elo, l_elo)
        l_expected = expected_score(l_elo, w_elo)

        w_new = update_elo(w_elo, w_expected, actual=1, k=K_FACTOR)
        l_new = update_elo(l_elo, l_expected, actual=0, k=K_FACTOR)

        ratings[winner_id] = w_new
        ratings[loser_id] = l_new
    else:
        winner_surface_elo_pre.append(np.nan)
        loser_surface_elo_pre.append(np.nan)

elo_matches["winner_surface_ELO"] = winner_surface_elo_pre
elo_matches["loser_surface_ELO"] = loser_surface_elo_pre

In [18]:
elo_matches[
    [
        "tourney_date",
        "surface",
        "winner_name",
        "loser_name",
        "winner_surface_ELO",
        "loser_surface_ELO",
    ]
].head(20)

,tourney_date,surface,winner_name,loser_name,winner_surface_ELO,loser_surface_ELO
0,2013-01-07,Hard,Yen Hsun Lu,Benoit Paire,1500.000000,1500.0
1,2013-01-07,Hard,Lukas Lacko,Paolo Lorenzi,1500.000000,1500.0
2,2013-01-07,Hard,Thomaz Bellucci,David Goffin,1500.000000,1500.0
3,2013-01-07,Hard,Igor Sijsling,Robin Haase,1500.000000,1500.0
4,2013-01-07,Hard,Gael Monfils,Benjamin Becker,1500.000000,1500.0
5,2013-01-07,Hard,Greg Jones,Jurgen Melzer,1500.000000,1500.0
6,2013-01-07,Hard,Brian Baker,Jerzy Janowicz,1500.000000,1500.0
7,2013-01-07,Hard,Jesse Levine,Daniel King Turner,1500.000000,1500.0
8,2013-01-07,Hard,Olivier Rochus,Albert Ramos,1500.000000,1500.0
9,2013-01-07,Hard,Xavier Malisse,Martin Klizan,1500.000000,1500.0


In [19]:
elo_matches[
    ["winner_surface_ELO", "loser_surface_ELO"]
].describe()

,winner_surface_ELO,loser_surface_ELO
count,29408.000000,29408.000000
mean,1617.377359,1562.850192
std,143.399833,109.225793
min,1311.818412,1310.899240
25%,1508.718983,1490.676756
50%,1582.540574,1532.224264
75%,1693.342180,1614.536596
max,2222.533962,2227.239003


In [20]:
elo_matches["winner_hELO"] = np.where(elo_matches["surface"] == "Hard", elo_matches["winner_surface_ELO"], np.nan)
elo_matches["loser_hELO"]  = np.where(elo_matches["surface"] == "Hard", elo_matches["loser_surface_ELO"], np.nan)

elo_matches["winner_cELO"] = np.where(elo_matches["surface"] == "Clay", elo_matches["winner_surface_ELO"], np.nan)
elo_matches["loser_cELO"]  = np.where(elo_matches["surface"] == "Clay", elo_matches["loser_surface_ELO"], np.nan)

elo_matches["winner_gELO"] = np.where(elo_matches["surface"] == "Grass", elo_matches["winner_surface_ELO"], np.nan)
elo_matches["loser_gELO"]  = np.where(elo_matches["surface"] == "Grass", elo_matches["loser_surface_ELO"], np.nan)

In [21]:
elo_matches[
    [
        "surface",
        "winner_name",
        "loser_name",
        "winner_ELO",
        "loser_ELO",
        "winner_surface_ELO",
        "loser_surface_ELO",
        "winner_hELO",
        "winner_cELO",
        "winner_gELO",
    ]
].head(20)

,surface,winner_name,loser_name,winner_ELO,loser_ELO,winner_surface_ELO,loser_surface_ELO,winner_hELO,winner_cELO,winner_gELO
0,Hard,Yen Hsun Lu,Benoit Paire,1500.000000,1500.0,1500.000000,1500.0,1500.000000,NaN,NaN
1,Hard,Lukas Lacko,Paolo Lorenzi,1500.000000,1500.0,1500.000000,1500.0,1500.000000,NaN,NaN
2,Hard,Thomaz Bellucci,David Goffin,1500.000000,1500.0,1500.000000,1500.0,1500.000000,NaN,NaN
3,Hard,Igor Sijsling,Robin Haase,1500.000000,1500.0,1500.000000,1500.0,1500.000000,NaN,NaN
4,Hard,Gael Monfils,Benjamin Becker,1500.000000,1500.0,1500.000000,1500.0,1500.000000,NaN,NaN
5,Hard,Greg Jones,Jurgen Melzer,1500.000000,1500.0,1500.000000,1500.0,1500.000000,NaN,NaN
6,Hard,Brian Baker,Jerzy Janowicz,1500.000000,1500.0,1500.000000,1500.0,1500.000000,NaN,NaN
7,Hard,Jesse Levine,Daniel King Turner,1500.000000,1500.0,1500.000000,1500.0,1500.000000,NaN,NaN
8,Hard,Olivier Rochus,Albert Ramos,1500.000000,1500.0,1500.000000,1500.0,1500.000000,NaN,NaN
9,Hard,Xavier Malisse,Martin Klizan,1500.000000,1500.0,1500.000000,1500.0,1500.000000,NaN,NaN


## 7.2 Merge Elo Features into the Player-Match Table

The match-level Elo ratings are merged into the long-format player-match dataset. Each player-match row receives the relevant pre-match overall Elo, career-high Elo, and surface-specific Elo corresponding to that player's role in the match (winner or loser).

In [22]:
elo_winner = elo_matches[
    [
        "tourney_date", "tourney_id", "match_num", "winner_id",
        "winner_ELO", "winner_ELOHigh",
        "winner_hELO", "winner_cELO", "winner_gELO"
    ]
].rename(columns={
    "winner_id": "player_id",
    "winner_ELO": "ELO",
    "winner_ELOHigh": "ELOHigh",
    "winner_hELO": "hELO",
    "winner_cELO": "cELO",
    "winner_gELO": "gELO",
})

elo_loser = elo_matches[
    [
        "tourney_date", "tourney_id", "match_num", "loser_id",
        "loser_ELO", "loser_ELOHigh",
        "loser_hELO", "loser_cELO", "loser_gELO"
    ]
].rename(columns={
    "loser_id": "player_id",
    "loser_ELO": "ELO",
    "loser_ELOHigh": "ELOHigh",
    "loser_hELO": "hELO",
    "loser_cELO": "cELO",
    "loser_gELO": "gELO",
})

elo_long = pd.concat([elo_winner, elo_loser], ignore_index=True)

elo_long = elo_long.sort_values(
    ["player_id", "tourney_date", "tourney_id", "match_num"]
).reset_index(drop=True)

print("elo_long shape:", elo_long.shape)
elo_long.head()

elo_long shape: (58816, 9)


,tourney_date,tourney_id,match_num,player_id,ELO,ELOHigh,hELO,cELO,gELO
0,2013-07-15,2013-414,6,100644,1500.000000,1500.0,NaN,1500.000000,NaN
1,2014-04-28,2014-308,4,100644,1484.916968,1500.0,NaN,1484.276247,NaN
2,2014-07-07,2014-321,8,100644,1471.325590,1500.0,NaN,1466.225731,NaN
3,2014-07-14,2014-414,8,100644,1458.153754,1500.0,NaN,1452.838535,NaN
4,2014-07-14,2014-414,24,100644,1476.198666,1500.0,NaN,1471.478154,NaN


In [23]:
df_long = df_long.merge(
    elo_long,
    on=["tourney_date", "tourney_id", "match_num", "player_id"],
    how="left"
)

print("df_long shape after Elo merge:", df_long.shape)

display(
    df_long[
        [
            "player_id", "player_name", "tourney_date", "surface",
            "ELO", "ELOHigh", "hELO", "cELO", "gELO"
        ]
    ].head(20)
)

df_long shape after Elo merge: (58816, 93)


,player_id,player_name,tourney_date,surface,ELO,ELOHigh,hELO,cELO,gELO
0,100644,Alexander Zverev,2013-07-15,Clay,1500.000000,1500.000000,NaN,1500.000000,NaN
1,100644,Alexander Zverev,2014-04-28,Clay,1484.916968,1500.000000,NaN,1484.276247,NaN
2,100644,Alexander Zverev,2014-07-07,Clay,1471.325590,1500.000000,NaN,1466.225731,NaN
3,100644,Alexander Zverev,2014-07-14,Clay,1458.153754,1500.000000,NaN,1452.838535,NaN
4,100644,Alexander Zverev,2014-07-14,Clay,1476.198666,1500.000000,NaN,1471.478154,NaN
5,100644,Alexander Zverev,2014-07-14,Clay,1496.774335,1500.000000,NaN,1491.241292,NaN
6,100644,Alexander Zverev,2014-07-14,Clay,1516.864109,1516.864109,NaN,1511.629398,NaN
7,100644,Alexander Zverev,2014-07-14,Clay,1532.320660,1532.320660,NaN,1527.793219,NaN
8,100644,Alexander Zverev,2014-07-21,Clay,1526.201909,1532.320660,NaN,1521.081472,NaN
9,100644,Alexander Zverev,2014-07-27,Clay,1509.652934,1532.320660,NaN,1506.260435,NaN


## 7.3 Construct Active Surface Elo

A unified surface-specific Elo column is created by selecting the appropriate surface Elo value for the current match surface.

In [24]:
df_long["SurfaceELO"] = np.where(
    df_long["surface"] == "Hard", df_long["hELO"],
    np.where(
        df_long["surface"] == "Clay", df_long["cELO"],
        np.where(df_long["surface"] == "Grass", df_long["gELO"], np.nan)
    )
)

display(
    df_long[
        [
            "player_name", "surface", "ELO", "hELO", "cELO", "gELO", "SurfaceELO"
        ]
    ].head(20)
)

,player_name,surface,ELO,hELO,cELO,gELO,SurfaceELO
0,Alexander Zverev,Clay,1500.000000,NaN,1500.000000,NaN,1500.000000
1,Alexander Zverev,Clay,1484.916968,NaN,1484.276247,NaN,1484.276247
2,Alexander Zverev,Clay,1471.325590,NaN,1466.225731,NaN,1466.225731
3,Alexander Zverev,Clay,1458.153754,NaN,1452.838535,NaN,1452.838535
4,Alexander Zverev,Clay,1476.198666,NaN,1471.478154,NaN,1471.478154
5,Alexander Zverev,Clay,1496.774335,NaN,1491.241292,NaN,1491.241292
6,Alexander Zverev,Clay,1516.864109,NaN,1511.629398,NaN,1511.629398
7,Alexander Zverev,Clay,1532.320660,NaN,1527.793219,NaN,1527.793219
8,Alexander Zverev,Clay,1526.201909,NaN,1521.081472,NaN,1521.081472
9,Alexander Zverev,Clay,1509.652934,NaN,1506.260435,NaN,1506.260435


In [25]:
print(
    df_long[
        ["ELO", "ELOHigh", "hELO", "cELO", "gELO", "SurfaceELO"]
    ].describe()
)

                ELO       ELOHigh          hELO          cELO         gELO  \
count  58816.000000  58816.000000  34022.000000  18070.000000  6724.000000   
mean    1626.232663   1694.655879   1607.336331   1575.356951  1542.628653   
std      152.227765    163.890053    142.543813    113.977999    80.970033   
min     1291.732288   1500.000000   1311.818412   1310.899240  1355.587278   
25%     1509.891587   1557.579756   1500.000000   1500.000000  1494.847920   
50%     1595.775732   1666.679757   1572.346374   1547.044593  1519.006566   
75%     1707.464091   1785.222758   1688.615012   1626.557229  1575.712245   
max     2297.185069   2297.185069   2227.239003   2106.777191  1980.788090   

         SurfaceELO  
count  58816.000000  
mean    1590.113776  
std      130.345435  
min     1310.899240  
25%     1500.000000  
50%     1554.207238  
75%     1653.169982  
max     2227.239003  


## 7.4 Construct Elo Trend and Volatility Features

Additional Elo-based features are created to capture recent change and medium-term stability in player strength. Specifically, 26-week Elo change measures medium-term improvement or decline, while 52-week Elo standard deviation measures rating volatility.

In [26]:
elo_history = elo_long[
    ["player_id", "tourney_date", "tourney_id", "match_num", "ELO"]
].copy()

elo_history = elo_history.sort_values(
    ["player_id", "tourney_date", "tourney_id", "match_num"]
).reset_index(drop=True)

elo_history.head()

,player_id,tourney_date,tourney_id,match_num,ELO
0,100644,2013-07-15,2013-414,6,1500.000000
1,100644,2014-04-28,2014-308,4,1484.916968
2,100644,2014-07-07,2014-321,8,1471.325590
3,100644,2014-07-14,2014-414,8,1458.153754
4,100644,2014-07-14,2014-414,24,1476.198666


In [27]:
elo_history["target_26w_date"] = elo_history["tourney_date"] - pd.Timedelta(weeks=26)

elo_history_26w = elo_history[["player_id", "tourney_date", "ELO"]].rename(
    columns={
        "tourney_date": "elo_date_26w",
        "ELO": "ELO_26w_ago",
    }
)

elo_change_parts = []

for pid, group in elo_history.groupby("player_id"):
    current_g = group.sort_values("target_26w_date").copy()
    history_g = elo_history_26w[
        elo_history_26w["player_id"] == pid
    ].sort_values("elo_date_26w").copy()

    merged_g = pd.merge_asof(
        current_g,
        history_g[["elo_date_26w", "ELO_26w_ago"]],
        left_on="target_26w_date",
        right_on="elo_date_26w",
        direction="backward"
    )

    elo_change_parts.append(merged_g)

elo_history = pd.concat(elo_change_parts, ignore_index=True)
elo_history = elo_history.sort_values(
    ["player_id", "tourney_date", "tourney_id", "match_num"]
).reset_index(drop=True)

elo_history["ELOChange26W"] = elo_history["ELO"] - elo_history["ELO_26w_ago"]

# 52-week Elo volatility
elo_std_parts = []

for pid, group in elo_history.groupby("player_id"):
    group = group.sort_values(["tourney_date", "tourney_id", "match_num"]).copy()
    group = group.set_index("tourney_date")

    group["ELOSTD52W"] = (
        group["ELO"]
        .rolling("365D", min_periods=2)
        .std()
    )

    group = group.reset_index()
    group["player_id"] = pid
    elo_std_parts.append(group)

elo_history = pd.concat(elo_std_parts, ignore_index=True)
elo_history = elo_history.sort_values(
    ["player_id", "tourney_date", "tourney_id", "match_num"]
).reset_index(drop=True)

display(
    elo_history[
        ["player_id", "tourney_date", "tourney_id", "match_num", "ELO", "ELOChange26W", "ELOSTD52W"]
    ].head(20)
)

,player_id,tourney_date,tourney_id,match_num,ELO,ELOChange26W,ELOSTD52W
0,100644,2013-07-15,2013-414,6,1500.000000,NaN,NaN
1,100644,2014-04-28,2014-308,4,1484.916968,-15.083032,10.665314
2,100644,2014-07-07,2014-321,8,1471.325590,-28.674410,14.343670
3,100644,2014-07-14,2014-414,8,1458.153754,-41.846246,17.970625
4,100644,2014-07-14,2014-414,24,1476.198666,-23.801334,15.599997
5,100644,2014-07-14,2014-414,36,1496.774335,-3.225665,15.896275
6,100644,2014-07-14,2014-414,42,1516.864109,16.864109,19.798812
7,100644,2014-07-14,2014-414,45,1532.320660,32.320660,24.505354
8,100644,2014-07-21,2014-439,1,1526.201909,26.201909,27.307535
9,100644,2014-07-27,2014-319,12,1509.652934,9.652934,25.985317


In [28]:
elo_feature_history = elo_history[
    ["player_id", "tourney_date", "tourney_id", "match_num", "ELOChange26W", "ELOSTD52W"]
].copy()

# Drop broken columns first if they already exist
for col in ["ELOChange26W", "ELOSTD52W"]:
    if col in df_long.columns:
        df_long = df_long.drop(columns=[col])

df_long = df_long.merge(
    elo_feature_history,
    on=["player_id", "tourney_date", "tourney_id", "match_num"],
    how="left"
)

print("df_long shape after corrected Elo feature merge:", df_long.shape)

display(
    df_long[
        [
            "player_name", "tourney_date", "tourney_id", "match_num",
            "ELO", "ELOHigh", "SurfaceELO", "ELOChange26W", "ELOSTD52W"
        ]
    ].head(20)
)

df_long shape after corrected Elo feature merge: (58816, 96)


,player_name,tourney_date,tourney_id,match_num,ELO,ELOHigh,SurfaceELO,ELOChange26W,ELOSTD52W
0,Alexander Zverev,2013-07-15,2013-414,6,1500.000000,1500.000000,1500.000000,NaN,NaN
1,Alexander Zverev,2014-04-28,2014-308,4,1484.916968,1500.000000,1484.276247,-15.083032,10.665314
2,Alexander Zverev,2014-07-07,2014-321,8,1471.325590,1500.000000,1466.225731,-28.674410,14.343670
3,Alexander Zverev,2014-07-14,2014-414,8,1458.153754,1500.000000,1452.838535,-41.846246,17.970625
4,Alexander Zverev,2014-07-14,2014-414,24,1476.198666,1500.000000,1471.478154,-23.801334,15.599997
5,Alexander Zverev,2014-07-14,2014-414,36,1496.774335,1500.000000,1491.241292,-3.225665,15.896275
6,Alexander Zverev,2014-07-14,2014-414,42,1516.864109,1516.864109,1511.629398,16.864109,19.798812
7,Alexander Zverev,2014-07-14,2014-414,45,1532.320660,1532.320660,1527.793219,32.320660,24.505354
8,Alexander Zverev,2014-07-21,2014-439,1,1526.201909,1532.320660,1521.081472,26.201909,27.307535
9,Alexander Zverev,2014-07-27,2014-319,12,1509.652934,1532.320660,1506.260435,9.652934,25.985317


In [29]:
df_long[
    ["ELO", "ELOHigh", "SurfaceELO", "ELOChange26W", "ELOSTD52W"]
].describe()

,ELO,ELOHigh,SurfaceELO,ELOChange26W,ELOSTD52W
count,58816.000000,58816.000000,58816.000000,54220.000000,57518.000000
mean,1626.232663,1694.655879,1590.113776,11.988646,33.718595
std,152.227765,163.890053,130.345435,66.567250,17.452451
min,1291.732288,1500.000000,1310.899240,-239.346482,0.282509
25%,1509.891587,1557.579756,1500.000000,-29.932151,21.472487
50%,1595.775732,1666.679757,1554.207238,5.478491,30.785547
75%,1707.464091,1785.222758,1653.169982,51.327398,42.519263
max,2297.185069,2297.185069,2227.239003,383.801078,150.428484


## 7.5 Save Intermediate Player-Match Feature Table

The long-format player-match dataset, including rolling performance, ranking, and Elo features, is saved as an intermediate output before constructing the final match-level modelling table.

In [30]:
df_long.to_csv(FEATURE_DIR / "player_match_features_full.csv", index=False)

print("Saved:")
print((FEATURE_DIR / "player_match_features_full.csv").resolve())

Saved:
/Users/baileyhorsley/Documents/C. University/3rd Year/Project/Dissertation_Project/data/features/player_match_features_full.csv


## 8. Construct Final Match-Level Modelling Dataset

The feature-engineering process so far has produced a long-format dataset containing one row per player per match. For predictive modelling, this is transformed into a match-level dataset with one row per match.

To avoid target leakage, players are assigned to a neutral A/B ordering rather than winner/loser ordering. Within each match, the two player rows are ordered by player ID and labelled as player A and player B. The target variable is then defined as whether player A won the match.

In [31]:
match_key = ["tourney_date", "tourney_id", "match_num"]

print("Rows in df_long:", df_long.shape[0])
print("Unique matches in df_long:", df_long[match_key].drop_duplicates().shape[0])
print("Expected rows per match should be 2.")

Rows in df_long: 58816
Unique matches in df_long: 29408
Expected rows per match should be 2.


In [32]:
match_counts = df_long.groupby(match_key).size()

print(match_counts.value_counts().sort_index())

# Keep only standard two-player matches
valid_match_keys = match_counts[match_counts == 2].index
df_model_base = df_long.set_index(match_key).loc[valid_match_keys].reset_index()

print("Rows after keeping only two-player matches:", df_model_base.shape[0])
print("Unique matches after filter:", df_model_base[match_key].drop_duplicates().shape[0])

2    29408
Name: count, dtype: int64
Rows after keeping only two-player matches: 58816
Unique matches after filter: 29408


### 8.1 Assign Neutral Player A / Player B Ordering

Within each match, the two player rows are sorted by player ID and assigned as player A and player B. This creates a neutral ordering that does not encode the match outcome into the row structure.

In [33]:
df_model_base = df_model_base.sort_values(match_key + ["player_id"]).reset_index(drop=True)

df_model_base["player_order"] = df_model_base.groupby(match_key).cumcount()

print(df_model_base["player_order"].value_counts())

player_a = df_model_base[df_model_base["player_order"] == 0].copy()
player_b = df_model_base[df_model_base["player_order"] == 1].copy()

print("Player A rows:", player_a.shape)
print("Player B rows:", player_b.shape)

cols_to_keep_unprefixed = set(match_key)

player_a = player_a.rename(
    columns={col: f"A_{col}" for col in player_a.columns if col not in cols_to_keep_unprefixed}
)

player_b = player_b.rename(
    columns={col: f"B_{col}" for col in player_b.columns if col not in cols_to_keep_unprefixed}
)

df_model = player_a.merge(player_b, on=match_key, how="inner")

print("Final match-level shape:", df_model.shape)
df_model.head()

player_order
0    29408
1    29408
Name: count, dtype: int64
Player A rows: (29408, 97)
Player B rows: (29408, 97)
Final match-level shape: (29408, 191)


,tourney_date,tourney_id,match_num,A_tourney_name,A_surface,A_draw_size,A_tourney_level,A_score,A_best_of,A_round,...,B_WeekSinceCareerHigh,B_ELO,B_ELOHigh,B_hELO,B_cELO,B_gELO,B_SurfaceELO,B_ELOChange26W,B_ELOSTD52W,B_player_order
0,2013-01-07,2013-301,1,Auckland,Hard,28,A,6-3 2-6 6-2,3,R32,...,8.0,1500.0,1500.0,1500.0,NaN,NaN,1500.0,NaN,NaN,1
1,2013-01-07,2013-301,2,Auckland,Hard,28,A,6-3 6-3,3,R32,...,0.0,1500.0,1500.0,1500.0,NaN,NaN,1500.0,NaN,NaN,1
2,2013-01-07,2013-301,3,Auckland,Hard,28,A,7-6(5) 6-2,3,R32,...,10.0,1500.0,1500.0,1500.0,NaN,NaN,1500.0,NaN,NaN,1
3,2013-01-07,2013-301,4,Auckland,Hard,28,A,3-6 6-3 7-5,3,R32,...,9.0,1500.0,1500.0,1500.0,NaN,NaN,1500.0,NaN,NaN,1
4,2013-01-07,2013-301,5,Auckland,Hard,28,A,6-7(2) 6-3 6-4,3,R32,...,69.0,1500.0,1500.0,1500.0,NaN,NaN,1500.0,NaN,NaN,1


### 8.2 Create Target Variable

The target variable is defined as whether player A won the match. Since the A/B ordering is neutral and independent of outcome, this provides a valid binary classification target for match prediction.

In [34]:
df_model["target"] = df_model["A_won"].astype(int)

print(df_model["target"].value_counts())
print(df_model["target"].value_counts(normalize=True))

target
1    14945
0    14463
Name: count, dtype: int64
target
1    0.508195
0    0.491805
Name: proportion, dtype: float64


### 8.3 Create Difference Features

To reflect the comparative nature of tennis matches, selected ranking and rating-based features are converted into player-difference variables of the form A minus B.

In [35]:
df_model["RankDifference"] = df_model["A_CurrentRanking"] - df_model["B_CurrentRanking"]
df_model["ELODifference"] = df_model["A_ELO"] - df_model["B_ELO"]
df_model["SurfaceELODifference"] = df_model["A_SurfaceELO"] - df_model["B_SurfaceELO"]
df_model["RankChange26W_Diff"] = df_model["A_RankChange26W"] - df_model["B_RankChange26W"]
df_model["RankSTD52W_Diff"] = df_model["A_RankSTD52W"] - df_model["B_RankSTD52W"]
df_model["ELOChange26W_Diff"] = df_model["A_ELOChange26W"] - df_model["B_ELOChange26W"]
df_model["ELOSTD52W_Diff"] = df_model["A_ELOSTD52W"] - df_model["B_ELOSTD52W"]

### 8.4 Inspect Final Match-Level Dataset

The final modelling dataset is inspected to confirm that each row contains one match, a valid binary target, and the engineered player-level and difference-based features required for modelling.

In [36]:
print("Final modelling dataset shape:", df_model.shape)

display(
    df_model[
        [
            "tourney_date", "tourney_id", "match_num",
            "A_player_name", "B_player_name",
            "target",
            "A_CurrentRanking", "B_CurrentRanking", "RankDifference",
            "A_ELO", "B_ELO", "ELODifference",
            "A_SurfaceELO", "B_SurfaceELO", "SurfaceELODifference"
        ]
    ].head(20)
)

Final modelling dataset shape: (29408, 199)


,tourney_date,tourney_id,match_num,A_player_name,B_player_name,target,A_CurrentRanking,B_CurrentRanking,RankDifference,A_ELO,B_ELO,ELODifference,A_SurfaceELO,B_SurfaceELO,SurfaceELODifference
0,2013-01-07,2013-301,1,Yen Hsun Lu,Benoit Paire,1,60.0,43.0,17.0,1500.000000,1500.0,0.000000,1500.000000,1500.0,0.000000
1,2013-01-07,2013-301,2,Paolo Lorenzi,Lukas Lacko,0,57.0,48.0,9.0,1500.000000,1500.0,0.000000,1500.000000,1500.0,0.000000
2,2013-01-07,2013-301,3,Thomaz Bellucci,David Goffin,1,32.0,50.0,-18.0,1500.000000,1500.0,0.000000,1500.000000,1500.0,0.000000
3,2013-01-07,2013-301,4,Robin Haase,Igor Sijsling,0,54.0,70.0,-16.0,1500.000000,1500.0,0.000000,1500.000000,1500.0,0.000000
4,2013-01-07,2013-301,5,Benjamin Becker,Gael Monfils,0,65.0,99.0,-34.0,1500.000000,1500.0,0.000000,1500.000000,1500.0,0.000000
5,2013-01-07,2013-301,6,Jurgen Melzer,Greg Jones,0,29.0,373.0,-344.0,1500.000000,1500.0,0.000000,1500.000000,1500.0,0.000000
6,2013-01-07,2013-301,7,Brian Baker,Jerzy Janowicz,1,59.0,26.0,33.0,1500.000000,1500.0,0.000000,1500.000000,1500.0,0.000000
7,2013-01-07,2013-301,8,Daniel King Turner,Jesse Levine,0,376.0,104.0,272.0,1500.000000,1500.0,0.000000,1500.000000,1500.0,0.000000
8,2013-01-07,2013-301,9,Olivier Rochus,Albert Ramos,1,89.0,51.0,38.0,1500.000000,1500.0,0.000000,1500.000000,1500.0,0.000000
9,2013-01-07,2013-301,10,Xavier Malisse,Martin Klizan,1,62.0,30.0,32.0,1500.000000,1500.0,0.000000,1500.000000,1500.0,0.000000


In [37]:
print("Target balance:")
print(df_model["target"].value_counts())
print(df_model["target"].value_counts(normalize=True))

important_model_cols = [
    "target",
    "A_CurrentRanking", "B_CurrentRanking",
    "A_ELO", "B_ELO",
    "A_SurfaceELO", "B_SurfaceELO",
    "RankDifference", "ELODifference", "SurfaceELODifference"
]

print("\nMissing values in key modelling columns:")
print(df_model[important_model_cols].isnull().sum())

Target balance:
target
1    14945
0    14463
Name: count, dtype: int64
target
1    0.508195
0    0.491805
Name: proportion, dtype: float64

Missing values in key modelling columns:
target                   0
A_CurrentRanking         0
B_CurrentRanking        14
A_ELO                    0
B_ELO                    0
A_SurfaceELO             0
B_SurfaceELO             0
RankDifference          14
ELODifference            0
SurfaceELODifference     0
dtype: int64


In [38]:
df_model = df_model.dropna(subset=["A_CurrentRanking", "B_CurrentRanking", "RankDifference"]).copy()

print("Shape after dropping rows with missing ranking features:", df_model.shape)

print("\nMissing values in key modelling columns after drop:")
important_model_cols = [
    "target",
    "A_CurrentRanking", "B_CurrentRanking",
    "A_ELO", "B_ELO",
    "A_SurfaceELO", "B_SurfaceELO",
    "RankDifference", "ELODifference", "SurfaceELODifference"
]
print(df_model[important_model_cols].isnull().sum())

Shape after dropping rows with missing ranking features: (29394, 199)

Missing values in key modelling columns after drop:
target                  0
A_CurrentRanking        0
B_CurrentRanking        0
A_ELO                   0
B_ELO                   0
A_SurfaceELO            0
B_SurfaceELO            0
RankDifference          0
ELODifference           0
SurfaceELODifference    0
dtype: int64


### 8.5 Save Final Match-Level Feature Dataset

The final match-level modelling dataset is saved for use in training, validation, and SHAP analysis.

In [39]:
df_model.to_csv(FEATURE_DIR / "match_level_features.csv", index=False)

print("Saved:")
print((FEATURE_DIR / "match_level_features.csv").resolve())

Saved:
/Users/baileyhorsley/Documents/C. University/3rd Year/Project/Dissertation_Project/data/features/match_level_features.csv
